# 03 | Competing first outcomes following drought exit
Regenerate recovery/relapse/censor outcomes from deterministic SGI and newly identified drought runs, and fit cumulative incidence and cluster-robust cause-specific Cox models. First non-drought month is month 1.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import norm, spearmanr
BASE = Path.cwd() if (Path.cwd()/'data_input').exists() else Path.cwd().parent
assert (BASE/'data_input/monthly_heads_m_asl.csv').exists(), 'Open from repo root or notebooks/'
from statsmodels.duration.hazard_regression import PHReg
sgi=pd.read_csv(BASE/'data_processed/monthly_sgi.csv'); ev=pd.read_csv(BASE/'data_processed/drought_events.csv');ref=pd.read_csv(BASE/'data_processed/recovery_events.csv');mem=pd.read_csv(BASE/'data_processed/groundwater_memory.csv')


In [ ]:
rows=[]
series={(well,sc):g.sort_values('month').reset_index(drop=True) for well,g in sgi.groupby('well') for sc in [3,6,12]}
for e in ev.itertuples(index=False):
 sc=e.SGI_scale;well=e.Well;g=series[(well,sc)];v=g[f'SGI_{sc}'].to_numpy();j=np.flatnonzero(g.month.to_numpy()==e.End)[0];k=j+1
 eligible=k<len(v) and np.isfinite(v[k]) and v[k]>-1
 r={'scale':sc,'well':well,'group':e.Group,'start':e.Start,'end':e.End,'recovery_eligible':int(eligible),'outcome':None,'time_months':np.nan,'drought_duration':e.Observed_duration_months,'drought_min_sgi':e.Minimum_SGI}
 if eligible:
  for ix in range(k,len(v)):
   t=ix-k+1
   if not np.isfinite(v[ix]):r.update(outcome='censored',time_months=max(1,t-1));break
   if v[ix]>=0:r.update(outcome='recovered',time_months=t);break
   if v[ix]<=-1:r.update(outcome='relapse',time_months=t);break
  if r['outcome'] is None:r.update(outcome='censored',time_months=len(v)-k)
 rows.append(r)
calc=pd.DataFrame(rows)
assert len(calc)==len(ref)
keys=['scale','well','start','end']
ck=calc.merge(ref[keys+['outcome','recovery_eligible','time_months']],on=keys,suffixes=('_calc','_ref'),validate='one_to_one')
assert (ck.recovery_eligible_calc==ck.recovery_eligible_ref).all()
assert (ck.outcome_calc.fillna('NA')==ck.outcome_ref.fillna('NA')).all()
assert np.allclose(ck.time_months_calc,ck.time_months_ref,equal_nan=True)
print('Eligible:',calc.query('recovery_eligible==1').groupby('scale').size().to_dict())


In [ ]:
def cif(data,horizon):
 t=data.time_months.to_numpy(float);out=data.outcome.to_numpy();S=1.;Frec=0.;Frel=0.
 for k in np.sort(np.unique(t[t<=horizon])):
  risk=np.sum(t>=k);nrec=np.sum((t==k)&(out=='recovered'));nrel=np.sum((t==k)&(out=='relapse'))
  Frec+=S*nrec/risk;Frel+=S*nrel/risk;S*=1-(nrec+nrel)/risk
 return Frec,Frel
q=calc.query('recovery_eligible==1 and group in ["D","M"]').merge(mem[['well','detrended_M12']],on='well')
for sc in [3,6,12]:
 for group in ['D','M']:
  d=q.query('scale==@sc and group==@group');print(sc,group,'12m recovery/relapse CIF:',tuple(round(a,4) for a in cif(d,12)))


In [ ]:
sd=mem.query('group in ["D","M"]').detrended_M12.std(ddof=1)
for sc in [3,6,12]:
 d=q.query('scale==@sc').copy();d['event']=(d.outcome=='recovered').astype(int)
 d['memory_sd']=d.detrended_M12/sd;d['log_duration']=np.log(d.drought_duration);d['is_dune']=(d.group=='D').astype(int)
 fit=PHReg(d.time_months,d[['memory_sd','log_duration','drought_min_sgi','is_dune']],status=d.event,ties='efron').fit(groups=d.well)
 beta,se=fit.params[0],fit.bse[0]
 print(f'SGI-{sc}: n={len(d)}, events={d.event.sum()}, HR={np.exp(beta):.4f}, CI=({np.exp(beta-1.96*se):.4f}, {np.exp(beta+1.96*se):.4f})')


These results are based on the deterministic SGI convention and may differ slightly from manuscript v15 until the figures and text are reconciled.